## requirement usage

This notebook introduces `requirement` usage and `assert satisfy ... by ...`; after running it you can apply a requirement definition to named design candidates and record which ones satisfy it.

Chapter 2 defined `TimelyToast` as a requirement definition with a `Toaster` subject. A requirement definition describes *what* must hold; a requirement usage applies it to actual candidates. This notebook adds `requirement timely : TimelyToast;` and two assert-satisfy claims — one for the `nominal` variant (120 s) and one for `slow` (200 s).

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
}"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: assert satisfy against an undefined requirement reference
# raises "unresolved reference" at the assert-satisfy site.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Toaster { attribute cycleTime : Real default = 120.0; }
    part nominal : Toaster;
    part evidence { assert satisfy undefinedReq by nominal; }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
req_usage = model.find("ToasterDemo::timely")
assert req_usage is not None
print(f"requirement usage kind: {req_usage.kind}")

for e in model.query():
    d = e.as_dict()
    if d.get("@type") == "RequirementUsage":
        print(f"RequirementUsage: {d['qualifiedName']}")
conn.close()

`requirement timely : TimelyToast; part evidence { assert satisfy timely by nominal; assert satisfy timely by slow; }` is the A-F declaration; OpenSysML parses the satisfy relationships and registers them (O-S); `model.find()` returns the RequirementUsage symbol and `model.query()` lists it (E).

Try the chapter exercise in `exercises/ch03/exercise.ipynb`: declare a `TemperatureReq` usage and assert satisfy for your `nominal` and `hot` coffee maker candidates.